# NB6 — Deep Learning : Fine-tuning de Transformers et architectures pré-Transformers

## Contexte

Ce notebook représente la dernière étape de modélisation du projet de
classification de tweets disaster/non-disaster. Il fait suite aux notebooks
NB1 à NB5 qui ont exploré les approches classiques (TF-IDF, embeddings,
ensemblistes) et plafonnent autour de **F1 = 0.717**.

L'objectif est de **dépasser ce score** en exploitant les architectures de
deep learning, en particulier les **Transformers** qui ont révolutionné le
NLP depuis 2018 grâce à leur capacité à capturer le **contexte bidirectionnel**
des mots.

---

## Scores de référence à battre

| Notebook | Meilleur pipeline | F1 classe 1 |
|---|---|---|
| NB1 | P03_TFIDF_UniBi_LogReg | 0.710 |
| NB2 | P08_TFIDF_Word_LogReg_Balanced | 0.705 |
| NB3 | P12_Count_ComplementNB | 0.712 |
| NB4 | P19_GloVeTwitter_LinearSVC | 0.625 |
| NB5 | P24_Voting | 0.717 |
| **Référence globale** | **P24_Voting** | **0.717** |

---

## Modèles testés

Ce notebook compare **5 architectures de deep learning** appartenant à
deux générations différentes :

### Génération moderne — Transformers (2018+)
- **RoBERTa** (`roberta-base`) : version optimisée de BERT, pré-entraînée
  sur 160 GB de texte
- **BERTweet** (`vinai/bertweet-base`) : transformer spécialisé pour les
  tweets, pré-entraîné sur 850 millions de tweets anglais
- **DistilBERT** (`distilbert-base-uncased`) : version distillée et allégée
  de BERT (66M paramètres au lieu de 110M)

### Génération pré-Transformer (2014-2017)
- **BiLSTM** : réseau récurrent bidirectionnel qui lit la séquence dans les
  deux sens — état de l'art en NLP avant 2018
- **TextCNN** (Kim, 2014) : application des convolutions au texte, capture
  les n-grams via des filtres convolutifs parallèles

Cette diversité d'architectures permet de **mesurer l'apport réel des
Transformers** par rapport aux architectures qui les ont précédées.

---

## Méthodologie

### Fine-tuning des Transformers
Les modèles RoBERTa, BERTweet et DistilBERT sont déjà pré-entraînés sur des
milliards de mots. On ajoute simplement une **couche de classification**
(Linear 768 → 2 classes) au-dessus de l'encoder, puis on fine-tune le tout
sur notre dataset pendant 2-3 epochs avec un learning rate très faible
(2e-5) pour ne pas oublier les représentations apprises.

### Entraînement from-scratch (BiLSTM, TextCNN)
Ces modèles plus simples sont entraînés **de zéro** sur notre dataset,
sans pré-entraînement. Ils utilisent un **vocabulaire custom** construit
à partir du corpus.

### Environnement d'exécution
Le fine-tuning des Transformers nécessite un **GPU** pour des temps raisonnables :
ce notebook est exécuté sur **Google Colab avec GPU T4** (gratuit).

---

## Métriques évaluées

* **F1-score classe 1** (métrique principale)
* **Recall classe 1** (priorité pour un système d'alerte)
* **Precision classe 1**
* **Accuracy globale**
* **Balanced accuracy**

In [1]:
# Installation propre des versions compatibles
!pip install -q "numpy>=2.0" "transformers" "datasets" "accelerate"
print("✅ Installation propre terminée")

✅ Installation propre terminée


In [4]:
# CELLULE 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset

from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    balanced_accuracy_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device : {device}")

✅ Device : cuda


In [5]:
!pip uninstall torchvision -y -q
print("✅ torchvision désinstallé")

✅ torchvision désinstallé


In [6]:
# CELLULE 2 — Chargement données
train = pd.read_csv('/content/train_cleaned.csv')
test  = pd.read_csv('/content/test_cleaned.csv')
train['text_clean'] = train['text_clean'].fillna('')
test['text_clean']  = test['text_clean'].fillna('')
print(f"✅ Train : {train.shape}, Test : {test.shape}")

✅ Train : (8979, 17), Test : (2245, 17)


In [7]:
# CELLULE 3 — Configuration
MODEL_NAME    = 'vinai/bertweet-base'
TEXT_COL      = 'text_clean'
LABEL_COL     = 'target'
RANDOM_STATE  = 42
REFERENCE_F1  = 0.717
MAX_LENGTH    = 64
BATCH_SIZE    = 32
EPOCHS        = 2
LEARNING_RATE = 2e-5

X_train = train[TEXT_COL].tolist()
X_test  = test[TEXT_COL].tolist()
y_train = train[LABEL_COL].tolist()
y_test  = test[LABEL_COL].tolist()
print("✅ Config OK")

✅ Config OK


In [8]:
# CELLULE 4 — Tokenisation
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation = True,
        padding    = 'max_length',
        max_length = MAX_LENGTH,
    )

train_dataset = Dataset.from_dict({'text': X_train, 'label': y_train})
test_dataset  = Dataset.from_dict({'text': X_test,  'label': y_test})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function,  batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch',  columns=['input_ids', 'attention_mask', 'label'])

print(f"✅ Tokenisation terminée")

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Map:   0%|          | 0/8979 [00:00<?, ? examples/s]

Map:   0%|          | 0/2245 [00:00<?, ? examples/s]

✅ Tokenisation terminée


In [9]:
# CELLULE 5 — Chargement du modèle
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
).to(device)

print(f"✅ Modèle chargé sur {device}")
print(f"   Paramètres : {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modèle chargé sur cuda
   Paramètres : 134,901,506


In [10]:
# CELLULE 6 — Métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return {
        'f1'                : f1_score(labels, predictions),
        'accuracy'          : accuracy_score(labels, predictions),
        'recall'            : recall_score(labels, predictions),
        'precision'         : precision_score(labels, predictions),
        'balanced_accuracy' : balanced_accuracy_score(labels, predictions),
    }
print("✅ Métriques OK")

✅ Métriques OK


In [11]:
# CELLULE 7 — Configuration de l'entraînement
training_args = TrainingArguments(
    output_dir                  = './bertweet_results',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = 'none',
    seed                        = RANDOM_STATE,
    fp16                        = True,
)
print("✅ Training args OK")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training args OK


In [12]:
# CELLULE 8 — Entraînement
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)]
)

train_result = trainer.train()
print(f"\n✅ Fine-tuning terminé en {train_result.metrics['train_runtime']:.0f}s")

Epoch,Training Loss,Validation Loss,F1,Accuracy,Recall,Precision,Balanced Accuracy
1,0.276105,0.251630,0.733260,0.891759,0.804819,0.673387,0.858147
2,0.204604,0.235226,0.746575,0.901114,0.787952,0.709328,0.857364


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Fine-tuning terminé en 123s


In [13]:
# CELLULE 9 — Évaluation finale
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)

test_f1        = f1_score(y_test, y_pred)
test_recall    = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_accuracy  = accuracy_score(y_test, y_pred)
test_balanced  = balanced_accuracy_score(y_test, y_pred)
delta          = test_f1 - REFERENCE_F1

print(f"\n{'='*50}")
print(f"Résultats BERTweet :")
print(f"{'='*50}")
print(f"F1 classe 1        : {test_f1:.3f}")
print(f"Recall classe 1    : {test_recall:.3f}")
print(f"Precision classe 1 : {test_precision:.3f}")
print(f"Accuracy           : {test_accuracy:.3f}")
print(f"Balanced accuracy  : {test_balanced:.3f}")
print(f"\nDelta vs référence : {delta:+.3f} {'🟢' if delta > 0 else '🔴'}")

print(f"\n{classification_report(y_test, y_pred, target_names=['Non-disaster', 'Disaster'])}")


Résultats BERTweet :
F1 classe 1        : 0.747
Recall classe 1    : 0.788
Precision classe 1 : 0.709
Accuracy           : 0.901
Balanced accuracy  : 0.857

Delta vs référence : +0.030 🟢

              precision    recall  f1-score   support

Non-disaster       0.95      0.93      0.94      1830
    Disaster       0.71      0.79      0.75       415

    accuracy                           0.90      2245
   macro avg       0.83      0.86      0.84      2245
weighted avg       0.91      0.90      0.90      2245



In [14]:
# CELLULE 10 — Sauvegarde
model.save_pretrained('/content/bertweet_finetuned')
tokenizer.save_pretrained('/content/bertweet_finetuned')

results_nb6 = pd.DataFrame([{
    'pipeline'              : 'BERTweet_finetuned',
    'test_f1_class_1'       : test_f1,
    'test_recall_class_1'   : test_recall,
    'test_precision_class_1': test_precision,
    'test_accuracy'         : test_accuracy,
    'test_balanced_accuracy': test_balanced,
    'vs_reference'          : delta,
}])
results_nb6.to_csv('/content/NB6_bertweet_results.csv', index=False)

print("✅ Modèle et résultats sauvegardés dans /content/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle et résultats sauvegardés dans /content/


In [15]:
# Sauvegarde BERTweet et téléchargement local
!zip -r /content/bertweet_finetuned.zip /content/bertweet_finetuned/

from google.colab import files
files.download('/content/bertweet_finetuned.zip')
files.download('/content/NB6_bertweet_results.csv')
print("✅ BERTweet sauvegardé")

  adding: content/bertweet_finetuned/ (stored 0%)
  adding: content/bertweet_finetuned/model.safetensors (deflated 14%)
  adding: content/bertweet_finetuned/vocab.txt (deflated 50%)
  adding: content/bertweet_finetuned/added_tokens.json (stored 0%)
  adding: content/bertweet_finetuned/config.json (deflated 52%)
  adding: content/bertweet_finetuned/tokenizer_config.json (deflated 76%)
  adding: content/bertweet_finetuned/bpe.codes (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ BERTweet sauvegardé


In [16]:
# Initialisation du dictionnaire global des résultats
all_dl_results = []

# Ajoute le résultat BERTweet déjà obtenu
all_dl_results.append({
    'pipeline'              : 'BERTweet',
    'model'                 : 'vinai/bertweet-base',
    'test_f1_class_1'       : test_f1,      # 0.747
    'test_recall_class_1'   : test_recall,  # 0.788
    'test_precision_class_1': test_precision, # 0.709
    'test_accuracy'         : test_accuracy,   # 0.901
    'test_balanced_accuracy': test_balanced,   # 0.857
    'vs_reference'          : delta,           # +0.030
})
print(f"✅ BERTweet ajouté aux résultats globaux")

✅ BERTweet ajouté aux résultats globaux


In [17]:
# ── DISTILBERT — Changement de modèle ────────────────────────────────────────
MODEL_NAME = 'distilbert-base-uncased'
DISPLAY_NAME = 'DistilBERT'
print(f"🔄 Nouveau modèle : {MODEL_NAME}")

🔄 Nouveau modèle : distilbert-base-uncased


In [18]:
# ── TOKENISATION DISTILBERT ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation = True,
        padding    = 'max_length',
        max_length = MAX_LENGTH,
    )

train_dataset = Dataset.from_dict({'text': X_train, 'label': y_train})
test_dataset  = Dataset.from_dict({'text': X_test,  'label': y_test})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function,  batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch',  columns=['input_ids', 'attention_mask', 'label'])

print(f"✅ Tokenisation DistilBERT terminée")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8979 [00:00<?, ? examples/s]

Map:   0%|          | 0/2245 [00:00<?, ? examples/s]

✅ Tokenisation DistilBERT terminée


In [19]:
# ── CHARGEMENT DU MODÈLE DISTILBERT ──────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
).to(device)

print(f"✅ DistilBERT chargé sur {device}")
print(f"   Paramètres : {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ DistilBERT chargé sur cuda
   Paramètres : 66,955,010


In [20]:
# ── TRAINING ARGS DISTILBERT ─────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = './distilbert_results',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = 'none',
    seed                        = RANDOM_STATE,
    fp16                        = True,
)
print("✅ Training args OK")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training args OK


In [21]:
# ── ENTRAÎNEMENT DISTILBERT ──────────────────────────────────────────────────
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)]
)

train_result = trainer.train()
print(f"\n✅ Fine-tuning DistilBERT terminé en {train_result.metrics['train_runtime']:.0f}s")

Epoch,Training Loss,Validation Loss,F1,Accuracy,Recall,Precision,Balanced Accuracy
1,0.279824,0.245015,0.744731,0.902895,0.766265,0.724374,0.850072
2,0.197319,0.244162,0.728351,0.897996,0.739759,0.717290,0.836819


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Fine-tuning DistilBERT terminé en 55s


In [22]:
# ── ÉVALUATION DISTILBERT ────────────────────────────────────────────────────
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)

test_f1        = f1_score(y_test, y_pred)
test_recall    = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_accuracy  = accuracy_score(y_test, y_pred)
test_balanced  = balanced_accuracy_score(y_test, y_pred)
delta          = test_f1 - REFERENCE_F1

print(f"\n{'='*50}")
print(f"Résultats {DISPLAY_NAME} :")
print(f"{'='*50}")
print(f"F1 classe 1        : {test_f1:.3f}")
print(f"Recall classe 1    : {test_recall:.3f}")
print(f"Precision classe 1 : {test_precision:.3f}")
print(f"Accuracy           : {test_accuracy:.3f}")
print(f"Balanced accuracy  : {test_balanced:.3f}")
print(f"Delta vs référence : {delta:+.3f} {'🟢' if delta > 0 else '🔴'}")

# Ajout aux résultats globaux
all_dl_results.append({
    'pipeline'              : DISPLAY_NAME,
    'model'                 : MODEL_NAME,
    'test_f1_class_1'       : test_f1,
    'test_recall_class_1'   : test_recall,
    'test_precision_class_1': test_precision,
    'test_accuracy'         : test_accuracy,
    'test_balanced_accuracy': test_balanced,
    'vs_reference'          : delta,
})

# Sauvegarde
model.save_pretrained(f'/content/{DISPLAY_NAME}_finetuned')
tokenizer.save_pretrained(f'/content/{DISPLAY_NAME}_finetuned')
print(f"\n✅ {DISPLAY_NAME} sauvegardé")


Résultats DistilBERT :
F1 classe 1        : 0.745
Recall classe 1    : 0.766
Precision classe 1 : 0.724
Accuracy           : 0.903
Balanced accuracy  : 0.850
Delta vs référence : +0.028 🟢


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DistilBERT sauvegardé


In [23]:
# ── ROBERTA — Changement de modèle ───────────────────────────────────────────
MODEL_NAME   = 'roberta-base'
DISPLAY_NAME = 'RoBERTa'
print(f"🔄 Nouveau modèle : {MODEL_NAME}")

🔄 Nouveau modèle : roberta-base


In [24]:
# ── TOKENISATION ROBERTA ─────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation = True,
        padding    = 'max_length',
        max_length = MAX_LENGTH,
    )

train_dataset = Dataset.from_dict({'text': X_train, 'label': y_train})
test_dataset  = Dataset.from_dict({'text': X_test,  'label': y_test})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function,  batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch',  columns=['input_ids', 'attention_mask', 'label'])

print(f"✅ Tokenisation RoBERTa terminée")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/8979 [00:00<?, ? examples/s]

Map:   0%|          | 0/2245 [00:00<?, ? examples/s]

✅ Tokenisation RoBERTa terminée


In [25]:
# ── CHARGEMENT ROBERTA ───────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
).to(device)
print(f"✅ RoBERTa chargé : {sum(p.numel() for p in model.parameters()):,} paramètres")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ RoBERTa chargé : 124,647,170 paramètres


In [26]:
# ── TRAINING ARGS ROBERTA ────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = './roberta_results',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = 'none',
    seed                        = RANDOM_STATE,
    fp16                        = True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [27]:
# ── ENTRAÎNEMENT ROBERTA ─────────────────────────────────────────────────────
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)]
)

train_result = trainer.train()
print(f"\n✅ RoBERTa terminé en {train_result.metrics['train_runtime']:.0f}s")

Epoch,Training Loss,Validation Loss,F1,Accuracy,Recall,Precision,Balanced Accuracy
1,0.277845,0.247815,0.747519,0.897996,0.816867,0.689024,0.866630
2,0.190695,0.224376,0.770810,0.910468,0.814458,0.731602,0.873349


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ RoBERTa terminé en 128s


In [28]:
# ── ÉVALUATION ROBERTA ───────────────────────────────────────────────────────
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)

test_f1        = f1_score(y_test, y_pred)
test_recall    = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_accuracy  = accuracy_score(y_test, y_pred)
test_balanced  = balanced_accuracy_score(y_test, y_pred)
delta          = test_f1 - REFERENCE_F1

print(f"\n{'='*50}")
print(f"Résultats {DISPLAY_NAME} :")
print(f"{'='*50}")
print(f"F1 classe 1        : {test_f1:.3f}")
print(f"Recall classe 1    : {test_recall:.3f}")
print(f"Precision classe 1 : {test_precision:.3f}")
print(f"Accuracy           : {test_accuracy:.3f}")
print(f"Balanced accuracy  : {test_balanced:.3f}")
print(f"Delta vs référence : {delta:+.3f} {'🟢' if delta > 0 else '🔴'}")

all_dl_results.append({
    'pipeline'              : DISPLAY_NAME,
    'model'                 : MODEL_NAME,
    'test_f1_class_1'       : test_f1,
    'test_recall_class_1'   : test_recall,
    'test_precision_class_1': test_precision,
    'test_accuracy'         : test_accuracy,
    'test_balanced_accuracy': test_balanced,
    'vs_reference'          : delta,
})

model.save_pretrained(f'/content/{DISPLAY_NAME}_finetuned')
tokenizer.save_pretrained(f'/content/{DISPLAY_NAME}_finetuned')
print(f"\n✅ {DISPLAY_NAME} sauvegardé")


Résultats RoBERTa :
F1 classe 1        : 0.771
Recall classe 1    : 0.814
Precision classe 1 : 0.732
Accuracy           : 0.910
Balanced accuracy  : 0.873
Delta vs référence : +0.054 🟢


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ RoBERTa sauvegardé


In [29]:
# ── BiLSTM — IMPORTS ET PRÉPARATION ──────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import re

DISPLAY_NAME = 'BiLSTM'
print(f"🔄 Modèle : {DISPLAY_NAME}")

🔄 Modèle : BiLSTM


In [30]:
# ── CONSTRUCTION DU VOCABULAIRE ──────────────────────────────────────────────
# Contrairement à BERT qui a son propre tokeniseur, ici on construit
# notre propre vocabulaire à partir du corpus

def simple_tokenize(text):
    """Tokenisation simple : minuscules + mots seulement"""
    text = text.lower()
    return re.findall(r'\b\w+\b', text)

# Construction du vocabulaire à partir du train
print("Construction du vocabulaire...")
word_counts = Counter()
for text in X_train:
    word_counts.update(simple_tokenize(text))

# Vocabulaire : mots apparaissant au moins 2 fois
VOCAB_SIZE = 10000
MIN_FREQ   = 2

# 0 = padding, 1 = unknown (mots inconnus)
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.most_common(VOCAB_SIZE - 2):
    if count >= MIN_FREQ:
        word2idx[word] = len(word2idx)

VOCAB_SIZE_REAL = len(word2idx)
print(f"✅ Vocabulaire : {VOCAB_SIZE_REAL} mots")

Construction du vocabulaire...
✅ Vocabulaire : 8615 mots


In [31]:
# ── DATASET PYTORCH ──────────────────────────────────────────────────────────
class TweetDataset(TorchDataset):
    """
    Dataset PyTorch pour tweets. Convertit chaque tweet en séquence
    d'indices dans le vocabulaire.
    """
    def __init__(self, texts, labels, word2idx, max_length=64):
        self.texts      = texts
        self.labels     = labels
        self.word2idx   = word2idx
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = simple_tokenize(self.texts[idx])
        # Conversion mots → indices (UNK si mot inconnu)
        indices = [self.word2idx.get(w, 1) for w in tokens[:self.max_length]]
        # Padding à max_length
        indices = indices + [0] * (self.max_length - len(indices))
        return (
            torch.tensor(indices, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# Création des DataLoaders
MAX_LEN = 64
train_ds = TweetDataset(X_train, y_train, word2idx, MAX_LEN)
test_ds  = TweetDataset(X_test,  y_test,  word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f"✅ Datasets créés — Train : {len(train_ds)}, Test : {len(test_ds)}")

✅ Datasets créés — Train : 8979, Test : 2245


In [32]:
# ── ARCHITECTURE BiLSTM ──────────────────────────────────────────────────────
class BiLSTMClassifier(nn.Module):
    """
    Architecture BiLSTM pour classification de texte :
    1. Embedding : convertit chaque mot en vecteur dense (128d)
    2. BiLSTM   : RNN bidirectionnel — lit la séquence dans les deux sens
    3. Pooling  : moyenne des sorties du BiLSTM
    4. Dropout  : régularisation
    5. Linear   : classification (2 classes)
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_classes=2, dropout=0.3):
        super().__init__()

        # Couche d'embedding : transforme indices en vecteurs denses
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # BiLSTM : bidirectionnel (lit la phrase dans les 2 sens)
        # hidden_dim*2 en sortie car concatène forward + backward
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first   = True,
            bidirectional = True,
            num_layers    = 1
        )

        self.dropout = nn.Dropout(dropout)
        # *2 car bidirectionnel
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded     = self.embedding(x)              # [batch, seq, embed_dim]
        lstm_out, _  = self.lstm(embedded)             # [batch, seq, hidden*2]
        pooled       = torch.mean(lstm_out, dim=1)     # [batch, hidden*2]
        dropped      = self.dropout(pooled)
        logits       = self.fc(dropped)                # [batch, num_classes]
        return logits

# Création du modèle
model = BiLSTMClassifier(
    vocab_size  = VOCAB_SIZE_REAL,
    embed_dim   = 128,
    hidden_dim  = 128,
    num_classes = 2,
    dropout     = 0.3
).to(device)

print(f"✅ BiLSTM créé")
print(f"   Paramètres : {sum(p.numel() for p in model.parameters()):,}")

✅ BiLSTM créé
   Paramètres : 1,367,426


In [33]:
# ── ENTRAÎNEMENT BiLSTM ──────────────────────────────────────────────────────
import torch.optim as optim
from tqdm import tqdm

# Poids des classes pour compenser le déséquilibre 81/19
class_weights = torch.tensor([
    len(y_train) / (2 * (len(y_train) - sum(y_train))),  # poids non-disaster
    len(y_train) / (2 * sum(y_train))                     # poids disaster
], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS_LSTM = 5
print(f"Entraînement BiLSTM ({EPOCHS_LSTM} epochs)...")

for epoch in range(EPOCHS_LSTM):
    # ── Phase d'entraînement ──────────────────────────────────────────────
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_LSTM}"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    # ── Phase d'évaluation ────────────────────────────────────────────────
    model.eval()
    test_correct = 0
    test_total   = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            test_correct += (outputs.argmax(1) == labels).sum().item()
            test_total   += labels.size(0)

    print(f"   Epoch {epoch+1} | "
          f"Train loss: {train_loss/len(train_loader):.4f} | "
          f"Train acc: {train_correct/train_total:.3f} | "
          f"Test acc: {test_correct/test_total:.3f}")

print(f"\n✅ Entraînement BiLSTM terminé")

Entraînement BiLSTM (5 epochs)...


Epoch 1/5: 100%|██████████| 141/141 [00:02<00:00, 54.00it/s]


   Epoch 1 | Train loss: 0.6116 | Train acc: 0.734 | Test acc: 0.854


Epoch 2/5: 100%|██████████| 141/141 [00:01<00:00, 87.29it/s] 


   Epoch 2 | Train loss: 0.4671 | Train acc: 0.829 | Test acc: 0.795


...

   Epoch 3 | Train loss: 0.3715 | Train acc: 0.869 | Test acc: 0.845


Epoch 4/5: 100%|██████████| 141/141 [00:01<00:00, 131.87it/s]


   Epoch 4 | Train loss: 0.2998 | Train acc: 0.894 | Test acc: 0.861


Epoch 5/5: 100%|██████████| 141/141 [00:00<00:00, 164.25it/s]


   Epoch 5 | Train loss: 0.2235 | Train acc: 0.921 | Test acc: 0.835

✅ Entraînement BiLSTM terminé


In [34]:
# ── ÉVALUATION BiLSTM ────────────────────────────────────────────────────────
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

y_pred = np.array(all_preds)

test_f1        = f1_score(y_test, y_pred)
test_recall    = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_accuracy  = accuracy_score(y_test, y_pred)
test_balanced  = balanced_accuracy_score(y_test, y_pred)
delta          = test_f1 - REFERENCE_F1

print(f"\n{'='*50}")
print(f"Résultats {DISPLAY_NAME} :")
print(f"{'='*50}")
print(f"F1 classe 1        : {test_f1:.3f}")
print(f"Recall classe 1    : {test_recall:.3f}")
print(f"Precision classe 1 : {test_precision:.3f}")
print(f"Accuracy           : {test_accuracy:.3f}")
print(f"Balanced accuracy  : {test_balanced:.3f}")
print(f"Delta vs référence : {delta:+.3f} {'🟢' if delta > 0 else '🔴'}")

all_dl_results.append({
    'pipeline'              : DISPLAY_NAME,
    'model'                 : 'BiLSTM (custom)',
    'test_f1_class_1'       : test_f1,
    'test_recall_class_1'   : test_recall,
    'test_precision_class_1': test_precision,
    'test_accuracy'         : test_accuracy,
    'test_balanced_accuracy': test_balanced,
    'vs_reference'          : delta,
})

# Sauvegarde
torch.save(model.state_dict(), f'/content/{DISPLAY_NAME}_model.pt')
print(f"\n✅ {DISPLAY_NAME} sauvegardé")


Résultats BiLSTM :
F1 classe 1        : 0.616
Recall classe 1    : 0.716
Precision classe 1 : 0.541
Accuracy           : 0.835
Balanced accuracy  : 0.789
Delta vs référence : -0.101 🔴

✅ BiLSTM sauvegardé


In [35]:
# ── CNN POUR TEXTE — ARCHITECTURE ────────────────────────────────────────────
DISPLAY_NAME = 'TextCNN'
print(f"🔄 Modèle : {DISPLAY_NAME}")

class TextCNN(nn.Module):
    """
    Architecture CNN pour classification de texte (Kim, 2014) :
    1. Embedding   : convertit mots en vecteurs (128d)
    2. Convolutions parallèles avec différentes tailles de filtres :
       - Filtre size 3 → capture les trigrammes
       - Filtre size 4 → capture les 4-grammes
       - Filtre size 5 → capture les 5-grammes
    3. Max pooling  : extrait la feature la plus importante par filtre
    4. Concatenation: combine les features des 3 tailles
    5. Dropout + Linear : classification
    """
    def __init__(self, vocab_size, embed_dim=128, num_filters=100,
                 filter_sizes=[3, 4, 5], num_classes=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Convolutions parallèles avec différentes tailles de filtres
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (fs, embed_dim))
            for fs in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(len(filter_sizes) * num_filters, num_classes)

    def forward(self, x):
        # x : [batch, seq_len]
        embedded = self.embedding(x)              # [batch, seq, embed]
        embedded = embedded.unsqueeze(1)          # [batch, 1, seq, embed]

        # Application des convolutions parallèles
        conved   = [torch.relu(conv(embedded)).squeeze(3) for conv in self.convs]
        # Max pooling sur la dimension temporelle
        pooled   = [torch.max(c, dim=2)[0] for c in conved]
        # Concaténation des features
        cat      = torch.cat(pooled, dim=1)        # [batch, num_filters*3]

        dropped  = self.dropout(cat)
        logits   = self.fc(dropped)                # [batch, num_classes]
        return logits

# Création du modèle
model = TextCNN(
    vocab_size   = VOCAB_SIZE_REAL,
    embed_dim    = 128,
    num_filters  = 100,
    filter_sizes = [3, 4, 5],
    num_classes  = 2,
    dropout      = 0.3
).to(device)

print(f"✅ TextCNN créé")
print(f"   Paramètres : {sum(p.numel() for p in model.parameters()):,}")

🔄 Modèle : TextCNN
✅ TextCNN créé
   Paramètres : 1,257,222


In [36]:
# ── ENTRAÎNEMENT TextCNN ─────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS_CNN = 5
print(f"Entraînement TextCNN ({EPOCHS_CNN} epochs)...")

for epoch in range(EPOCHS_CNN):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS_CNN}"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    model.eval()
    test_correct = 0
    test_total   = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            test_correct += (outputs.argmax(1) == labels).sum().item()
            test_total   += labels.size(0)

    print(f"   Epoch {epoch+1} | "
          f"Train loss: {train_loss/len(train_loader):.4f} | "
          f"Train acc: {train_correct/train_total:.3f} | "
          f"Test acc: {test_correct/test_total:.3f}")

print(f"\n✅ Entraînement TextCNN terminé")

Entraînement TextCNN (5 epochs)...


Epoch 1/5: 100%|██████████| 141/141 [00:02<00:00, 65.30it/s] 


   Epoch 1 | Train loss: 0.5876 | Train acc: 0.689 | Test acc: 0.841


Epoch 2/5: 100%|██████████| 141/141 [00:01<00:00, 122.18it/s]


   Epoch 2 | Train loss: 0.3631 | Train acc: 0.840 | Test acc: 0.853


...

   Epoch 3 | Train loss: 0.2309 | Train acc: 0.912 | Test acc: 0.852


Epoch 4/5: 100%|██████████| 141/141 [00:03<00:00, 46.57it/s]


   Epoch 4 | Train loss: 0.1474 | Train acc: 0.949 | Test acc: 0.862


Epoch 5/5: 100%|██████████| 141/141 [00:01<00:00, 116.29it/s]


   Epoch 5 | Train loss: 0.1113 | Train acc: 0.962 | Test acc: 0.873

✅ Entraînement TextCNN terminé


In [37]:
# ── ÉVALUATION TextCNN ───────────────────────────────────────────────────────
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

y_pred = np.array(all_preds)

test_f1        = f1_score(y_test, y_pred)
test_recall    = recall_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_accuracy  = accuracy_score(y_test, y_pred)
test_balanced  = balanced_accuracy_score(y_test, y_pred)
delta          = test_f1 - REFERENCE_F1

print(f"\n{'='*50}")
print(f"Résultats {DISPLAY_NAME} :")
print(f"{'='*50}")
print(f"F1 classe 1        : {test_f1:.3f}")
print(f"Recall classe 1    : {test_recall:.3f}")
print(f"Precision classe 1 : {test_precision:.3f}")
print(f"Accuracy           : {test_accuracy:.3f}")
print(f"Balanced accuracy  : {test_balanced:.3f}")
print(f"Delta vs référence : {delta:+.3f} {'🟢' if delta > 0 else '🔴'}")

all_dl_results.append({
    'pipeline'              : DISPLAY_NAME,
    'model'                 : 'TextCNN (custom)',
    'test_f1_class_1'       : test_f1,
    'test_recall_class_1'   : test_recall,
    'test_precision_class_1': test_precision,
    'test_accuracy'         : test_accuracy,
    'test_balanced_accuracy': test_balanced,
    'vs_reference'          : delta,
})

torch.save(model.state_dict(), f'/content/{DISPLAY_NAME}_model.pt')
print(f"\n✅ {DISPLAY_NAME} sauvegardé")


Résultats TextCNN :
F1 classe 1        : 0.615
Recall classe 1    : 0.549
Precision classe 1 : 0.699
Accuracy           : 0.873
Balanced accuracy  : 0.748
Delta vs référence : -0.102 🔴

✅ TextCNN sauvegardé


In [38]:
# ── SAUVEGARDE FINALE ─────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_dl_results).round(3)
results_df = results_df.sort_values('test_f1_class_1', ascending=False)

results_df.to_csv('/content/NB6_all_dl_results.csv', index=False)

print("=== Tableau comparatif final NB6 — Deep Learning ===")
display(results_df)

# Compression de tous les modèles
!zip -r /content/all_dl_models.zip /content/BERTweet_finetuned /content/DistilBERT_finetuned /content/RoBERTa_finetuned /content/BiLSTM_model.pt /content/TextCNN_model.pt

print("\n✅ Tous les modèles compressés")

=== Tableau comparatif final NB6 — Deep Learning ===


,pipeline,model,test_f1_class_1,test_recall_class_1,test_precision_class_1,test_accuracy,test_balanced_accuracy,vs_reference
2,RoBERTa,roberta-base,0.771,0.814,0.732,0.910,0.873,0.054
0,BERTweet,vinai/bertweet-base,0.747,0.788,0.709,0.901,0.857,0.030
1,DistilBERT,distilbert-base-uncased,0.745,0.766,0.724,0.903,0.850,0.028
3,BiLSTM,BiLSTM (custom),0.616,0.716,0.541,0.835,0.789,-0.101
4,TextCNN,TextCNN (custom),0.615,0.549,0.699,0.873,0.748,-0.102


	zip warning: name not matched: /content/BERTweet_finetuned
  adding: content/DistilBERT_finetuned/ (stored 0%)
  adding: content/DistilBERT_finetuned/model.safetensors (deflated 8%)
  adding: content/DistilBERT_finetuned/config.json (deflated 50%)
  adding: content/DistilBERT_finetuned/tokenizer_config.json (deflated 43%)
  adding: content/DistilBERT_finetuned/tokenizer.json (deflated 71%)
  adding: content/RoBERTa_finetuned/ (stored 0%)
  adding: content/RoBERTa_finetuned/model.safetensors (deflated 12%)
  adding: content/RoBERTa_finetuned/config.json (deflated 51%)
  adding: content/RoBERTa_finetuned/tokenizer_config.json (deflated 51%)
  adding: content/RoBERTa_finetuned/tokenizer.json (deflated 82%)
  adding: content/BiLSTM_model.pt (deflated 8%)
  adding: content/TextCNN_model.pt (deflated 7%)

✅ Tous les modèles compressés


In [39]:
# ── TÉLÉCHARGEMENT DES RÉSULTATS ──────────────────────────────────────────────
from google.colab import files

files.download('/content/NB6_all_dl_results.csv')
files.download('/content/all_dl_models.zip')

print("✅ Fichiers téléchargés sur ton ordi")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Fichiers téléchargés sur ton ordi


In [40]:
import os
for f in os.listdir('/content/'):
    if 'finetuned' in f or '.pt' in f:
        print(f"   → {f}")

   → BiLSTM_model.pt
   → TextCNN_model.pt
   → RoBERTa_finetuned
   → bertweet_finetuned
   → bertweet_finetuned.zip
   → DistilBERT_finetuned


In [41]:
# Compression et téléchargement de tous les modèles
!zip -r /content/RoBERTa_finetuned.zip /content/RoBERTa_finetuned/
!zip -r /content/DistilBERT_finetuned.zip /content/DistilBERT_finetuned/

from google.colab import files
files.download('/content/RoBERTa_finetuned.zip')
files.download('/content/DistilBERT_finetuned.zip')

print("✅ Téléchargements lancés")

  adding: content/RoBERTa_finetuned/ (stored 0%)
  adding: content/RoBERTa_finetuned/model.safetensors (deflated 12%)
  adding: content/RoBERTa_finetuned/config.json (deflated 51%)
  adding: content/RoBERTa_finetuned/tokenizer_config.json (deflated 51%)
  adding: content/RoBERTa_finetuned/tokenizer.json (deflated 82%)
  adding: content/DistilBERT_finetuned/ (stored 0%)
  adding: content/DistilBERT_finetuned/model.safetensors (deflated 8%)
  adding: content/DistilBERT_finetuned/config.json (deflated 50%)
  adding: content/DistilBERT_finetuned/tokenizer_config.json (deflated 43%)
  adding: content/DistilBERT_finetuned/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Téléchargements lancés


In [42]:
import os
csv_files = [f for f in os.listdir('/content/') if f.endswith('.csv')]
print("CSV disponibles :")
for f in sorted(csv_files):
    print(f"   → {f}")

CSV disponibles :
   → NB1_tuned_results.csv
   → NB2_tuned_results.csv
   → NB3_tuned_results.csv
   → NB4_tuned_results.csv
   → NB5_tuned_results.csv
   → NB6_all_dl_results.csv
   → NB6_bertweet_results.csv
   → test_cleaned.csv
   → train_cleaned.csv
